In [1]:
import polars as pl
import pandas as pd

import sys
sys.path.append('../../04_utils')

from utils import name_finder, low_context_name_finder

In [ ]:
# Define pathings
data_path = '../../01_data/'
out_path = '../../03_output/01_enriched_results/'

In [3]:
#Load data
verbs_df = pl.read_csv(data_path + 'DS1 verbs in -st.csv', separator= ',')

verbs_df.head()

Left,KWIC,Right
str,str,str
"""! - The legend of Artorias art…","""dost""","""thou say? </s><s> Wilt thou no…"
"""clan is thine own family. </s>…","""hast""","""thought on''t once more and wi…"
"""stay true. </s><s> Dare''st no…","""dost""","""cometh. </s><s> How fares ye? …"
"""it! </s><s> Make no attempt to…","""Hast""","""thou met Shiva? </s><s> A lad …"
"""for thee, take it. </s><s> May…","""dost""","""treat him with the same cautio…"


In [4]:
#Load Character master table
char_master_df = pl.read_csv(data_path + 'das_char_master.csv')\
                   .with_columns(pl.col('Character').str.replace(',','').alias('Character'))

char_master_df.head()

Character,Class,Age
str,str,str
"""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""ANASTACIA OF ASTORA""","""Low""","""Young"""
"""ANDRE OF ASTORA""","""Low""","""Old"""
"""BIG HAТ LOGAN""","""Low""","""Old"""
"""BLACKSMITH VAMOS""","""Low""","""Old"""


In [5]:
#Instantiate list with missing verbs
missing_verbs = ["Dare'st","Seekest","Didst","Need’st","Couldst","Hast","Shouldst","Needest","Com’st","Seest","Did’st",
                 "dare'st","seekest","didst","need’st","couldst","hast","shouldst","needest","com’st","seest","did’st"]

#Instantiate list with unneeded verbs
unneeded_verbs = ['feast','exist','cast','assist','lost','rest','entrust','trust','foist','resist','detest',
                  'Feast','Exist','Cast','Assist','Lost','Rest','Entrust','Trust','Foist','Resist','Detest']

In [6]:
# Open raw text to look for missing verbs
with open(data_path + 'Ds corpus input.txt', 'r', encoding='utf-8') as file:
        raw_text = file.read()  # Read the entire content into a single string
        # print(raw_text)

In [7]:
#Construct empty dataframe to input results
left_context = []
verbs = []
right_context = []
#Search for not found verbs
for missing in missing_verbs:
    idx = 0
    context_list = raw_text.split(missing)
    while idx + 1 < len(context_list):
        left_context.append(context_list[idx])
        verbs.append(missing)
        right_context.append(context_list[idx + 1][:300])
        idx += 1

# Construct dataframe with the found verbs and contexts, enrich with character name and format contexts to facilitate legibility
missing_verbs_df = pl.DataFrame().with_columns(pl.Series(left_context).alias('Left'),
                                                 pl.Series(verbs).alias('KWIC'),
                                                 pl.Series(right_context).alias('Right'))\
                     .with_columns(pl.col('Left').map_elements(name_finder, return_dtype=pl.Utf8).alias('Character'))\
                     .with_columns(pl.col('Left').str.replace_all('<s>','').str.replace_all('</s>',''),
                                   pl.col('Right').str.replace_all('<s>','').str.replace_all('</s>',''))\
                     .with_columns(pl.col('KWIC').str.to_lowercase().alias('KWIC'))\
                     .with_columns(pl.col('Character').forward_fill().alias('Character'))

missing_verbs_df


Left,KWIC,Right,Character
str,str,str,str
"""NARRATOR Yes, indeed. The Dark…","""dare'st""",""" not in any attempt to double-…","""ALVINA OF THE DARKROOT WOOD"""
"""NARRATOR Yes, indeed. The Dark…","""didst""",""" thou not see why Ariamis crea…","""CROSSBREED PRISCILLA"""
"""NARRATOR Yes, indeed. The Dark…","""couldst""",""" thou one more play the saviou…","""ELIZABETH KEEPER OF THE SANCTU…"
"""NARRATOR Yes, indeed. The Dark…","""hast""",""" thou met Shiva? → A lad comet…","""ALVINA OF THE DARKROOT WOOD"""
""" thou met Shiva? → A lad comet…","""hast""",""" thou reconsidered? Such that …","""ELIZABETH KEEPER OF THE SANCTU…"
…,…,…,…
""" taken a gander at it, But the…","""hast""",""" made thyself clear. And thou …","""OSWALD OF CARIM"""
""" made thyself clear. And thou …","""hast""",""" thou caus'd. Thou was weak in…","""OSWALD OF CARIM"""
"""NARRATOR Yes, indeed. The Dark…","""shouldst""",""" thine heart be changed, speak…","""ELIZABETH KEEPER OF THE SANCTU…"


In [8]:
missing_verbs_df.filter(pl.col('Character').is_null())

Left,KWIC,Right,Character
str,str,str,str


In [9]:
# Clean the raw text to facilitate matching with low_context_name_finder
punctuation = [',','.',';',':','!','?']

raw_text_clean = raw_text.replace('\n',' ').replace('…','...').replace('‘',"'").replace("' ","'").strip()
for punct in punctuation:
        raw_text_clean = raw_text_clean.replace(punct, punct + ' ').replace(punct,'')

raw_text_clean = raw_text_clean.split(' ')

raw_text_clean = [word for word in raw_text_clean if word != '']

raw_text_clean = ' '.join(raw_text_clean)

# raw_text_clean

In [10]:
# Extract the speaking character for each row, turn all KWIC lowercase to avoid redundant values.
verbs_df = verbs_df.filter(~pl.col('KWIC').is_in(missing_verbs)).filter(~pl.col('KWIC').is_in(unneeded_verbs))\
                   .with_columns(pl.col('Left').map_elements(lambda x: low_context_name_finder(x, raw_text_clean, 50), 
                                                             return_dtype=pl.Utf8).alias('Character'))\
                         .with_columns(pl.col('KWIC').str.to_lowercase().alias('KWIC'))\
                         .with_columns(pl.col('Left').str.replace_all('<s>','').str.replace_all('</s>','\n'),
                                       pl.col('Right').str.replace_all('<s>','').str.replace_all('</s>','\n'))\

# Concat with the missing verbs and trim contexts to facilitate legibility.
verbs_df = pl.concat([verbs_df, missing_verbs_df])\
             .with_columns(pl.col('Left').str.tail(300).alias('Left'),
                           pl.col('Right').str.head(300).alias('Right'))

verbs_df

Left,KWIC,Right,Character
str,str,str,str
"""s but a fairy tale. Have thi…","""dost""","""thou say? Wilt thou not join…","""ALVINA OF THE DARKROOT WOOD"""
"""s, never will we tolerate. O…","""dost""","""cometh. How fares ye? My h…","""ALVINA OF THE DARKROOT WOOD"""
"""Shiva? A lad cometh from the…","""dost""","""treat him with the same cautio…","""ALVINA OF THE DARKROOT WOOD"""
"""OSSBREED PRISCILLA Who art tho…","""dost""","""not belong. I beg of thee, p…","""CROSSBREED PRISCILLA"""
"""rom the plank, and hurry home.…","""dost""","""thee hurry toward thine death?…","""CROSSBREED PRISCILLA"""
…,…,…,…
"""h heh heh.. Thou art a friend.…","""hast""",""" made thyself clear. And thou …","""OSWALD OF CARIM"""
""" made thyself clear. And thou …","""hast""",""" thou caus'd. Thou was weak in…","""OSWALD OF CARIM"""
""" so imposing. But should thine…","""shouldst""",""" thine heart be changed, speak…","""ELIZABETH KEEPER OF THE SANCTU…"


In [11]:
verbs_df.filter(pl.col('Character').is_null())#['Left'].to_list()

Left,KWIC,Right,Character
str,str,str,str


In [12]:
#Add character class and age information from master table
verbs_df = verbs_df.join(char_master_df, on='Character', how = 'left')

verbs_df

Left,KWIC,Right,Character,Class,Age
str,str,str,str,str,str
"""s but a fairy tale. Have thi…","""dost""","""thou say? Wilt thou not join…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""s, never will we tolerate. O…","""dost""","""cometh. How fares ye? My h…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""Shiva? A lad cometh from the…","""dost""","""treat him with the same cautio…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""OSSBREED PRISCILLA Who art tho…","""dost""","""not belong. I beg of thee, p…","""CROSSBREED PRISCILLA""","""High""","""Old"""
"""rom the plank, and hurry home.…","""dost""","""thee hurry toward thine death?…","""CROSSBREED PRISCILLA""","""High""","""Old"""
…,…,…,…,…,…
"""h heh heh.. Thou art a friend.…","""hast""",""" made thyself clear. And thou …","""OSWALD OF CARIM""","""High""","""Old"""
""" made thyself clear. And thou …","""hast""",""" thou caus'd. Thou was weak in…","""OSWALD OF CARIM""","""High""","""Old"""
""" so imposing. But should thine…","""shouldst""",""" thine heart be changed, speak…","""ELIZABETH KEEPER OF THE SANCTU…","""High""","""Old"""


In [13]:
# Sanity check for extracted characters
verbs_df.to_pandas()['Character'].unique()

array(['ALVINA OF THE DARKROOT WOOD', 'CROSSBREED PRISCILLA',
       'DUSK OF OOLACILE', 'ELIZABETH KEEPER OF THE SANCTUARY',
       'HAWKEYE GOUGН', 'OSWALD OF CARIM', 'DARK SUN GWYNDOLIN',
       'GWYNEVERE PRINCESS OF SUNLIGHT'], dtype=object)

In [15]:
# Save enriched dataframe into a csv file.
verbs_df.write_csv(out_path + 'st_verbs_analysis.csv', separator= ';')